# Xử lý giá trị thiếu và ngoại lai

**Phạm vi notebook:** xử lý dữ liệu thiếu và kiểm tra ngoại lai cho `gia`, `dien_tich`; lưu một bản dữ liệu cleaned riêng.

**Quyết định:** giá/diện tích thiếu được điền median theo `vi_tri`, thiếu median nhóm thì dùng median toàn bộ; giữ giá trị gốc và cờ thiếu để truy vết. `so_phong_ngu` được giữ thiếu vì tin không nhắc đến số phòng không có nghĩa là số phòng bằng 0. Thiếu giá/diện tích có thể là **MNAR** do người bán không công khai; khác biệt thiếu giữa khu vực có thể gợi ý **MAR**. Không thể kết luận cơ chế chỉ từ dữ liệu quan sát.

Ngoại lai được xác định bằng phương pháp thống kê IQR và tiến hành loại bỏ hoàn toàn khỏi tập dữ liệu. Quyết định này nhằm đảm bảo các biểu đồ Khám phá dữ liệu (EDA) ở bước sau không bị méo lệch phân phối, đáp ứng Tiêu chí 1 Phần C.

## 1. Đọc và kiểm tra dữ liệu

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

# Tìm Project để notebook chạy được từ thư mục Project hoặc notebooks.
here = Path.cwd().resolve()
data_relative_path = Path('data') / 'processed' / 'cafeland_cleaned.csv'
project = next(
    (folder for folder in [here, *here.parents]
     if (folder / data_relative_path).is_file()),
    None,
)
if project is None:
    raise FileNotFoundError(f'Không tìm thấy {data_relative_path} từ {here}')

# Nạp dữ liệu
df_source = pd.read_csv(project / data_relative_path, encoding='utf-8-sig')

# --- BƯỚC SƠ CHẾ BẮT BUỘC TRƯỚC KHI TÍNH TOÁN ---

# 1. Chuẩn hóa tên cột (đảm bảo không bị lỗi chữ hoa/chữ thường hoặc dư khoảng trắng)
df_source.columns = df_source.columns.str.strip().str.lower()

# 2. Chốt chặn an toàn: Kiểm tra file từ Bài tập 2 có đủ cột để xử lý không
required_columns = {'gia', 'dien_tich', 'vi_tri', 'so_phong_ngu'}
if not required_columns.issubset(df_source.columns):
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(required_columns - set(df_source.columns))}')

# 3. Ép kiểu dữ liệu về Số thực (Float) để chuẩn bị cho thuật toán IQR
for column in ['gia', 'dien_tich', 'so_phong_ngu']:
    df_source[column] = pd.to_numeric(df_source[column], errors='coerce')

# 4. Xóa các chuỗi rỗng biến chúng thành NaN chuẩn của Pandas
df_source = df_source.replace(r'^\s*$', np.nan, regex=True)

print(f'Đã nạp thành công {len(df_source):,} dòng. Dữ liệu đã sẵn sàng để dọn dẹp ngoại lai.')

Đã nạp thành công 6,500 dòng. Dữ liệu đã sẵn sàng để dọn dẹp ngoại lai.


## 2. Đánh dấu và rà soát ngoại lai bằng IQR

IQR đánh dấu giá trị cần kiểm tra, không tự nó chứng minh giá trị sai. Xem tiêu đề và đường dẫn tin gốc trước khi quyết định loại/cắt; các tin dự án lớn có thể hợp lệ.

In [6]:
bounds = {}
outlier_report = []

for column in ['gia', 'dien_tich']:
    observed = df_source.loc[df_source[column] > 0, column]
    if observed.empty:
        raise ValueError(f'Không có giá trị dương để tính IQR cho {column}.')

    q1 = observed.quantile(0.25)
    q3 = observed.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    bounds[column] = (lower, upper)

    outlier = df_source[column].notna() & (
        (df_source[column] < lower) | (df_source[column] > upper)
    )
    outlier_report.append({
        'Biến': column,
        'Q1': q1,
        'Q3': q3,
        'Ngưỡng dưới': lower,
        'Ngưỡng trên': upper,
        'Số ngoại lai': int(outlier.sum()),
        'Tỷ lệ ngoại lai (%)': round(outlier.mean() * 100, 2),
    })

display(pd.DataFrame(outlier_report).set_index('Biến').round(2))

# Xem các tin có giá/diện tích lớn nhất để đặt ngoại lai vào ngữ cảnh thực tế.
for column in ['gia', 'dien_tich']:
    print(f'Các tin có {column} lớn nhất:')
    display(df_source.nlargest(5, column)[
        ['tieu_de', column, 'vi_tri', 'link_nguon']
    ])

,Q1,Q3,Ngưỡng dưới,Ngưỡng trên,Số ngoại lai,Tỷ lệ ngoại lai (%)
Biến,,,,,,
gia,3.741500e+09,1.850000e+10,-1.839625e+10,4.063775e+10,711,10.94
dien_tich,6.000000e+01,1.760000e+02,-1.140000e+02,3.500000e+02,599,9.22


Các tin có gia lớn nhất:


,tieu_de,gia,vi_tri,link_nguon
4389,"Quỹ Đất PTDA 5.1ha Trần Não & Mai Chí Thọ, Q2 ...",7.140000e+12,"An Khánh, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-51ha-t...
4346,"Quỹ Đất PTDA 1,809ha KĐT & KCN Tân Lâm, X. Hòa...",6.512000e+12,"Hòa Hội, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-1809ha...
4272,Chuyển nhượng nhà hẻm xe hơi 8m đường Hồng Bàn...,5.790000e+12,"Bình Thới, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/chuyen-nhuong-nha-h...
4457,"Quỹ Đất PTDA (70ha) MT Trần Đại Nghĩa, Bình Ch...",5.600000e+12,"Bình Lợi, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-70ha-m...
3424,"Quỹ Đất PTDA 36ha MT Nguyễn Hữu Thọ, Phước Kiể...",5.400000e+12,"Hiệp Phước, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-36ha-m...


Các tin có dien_tich lớn nhất:


,tieu_de,dien_tich,vi_tri,link_nguon
4346,"Quỹ Đất PTDA 1,809ha KĐT & KCN Tân Lâm, X. Hòa...",18090000.0,"Hòa Hội, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-1809ha...
4457,"Quỹ Đất PTDA (70ha) MT Trần Đại Nghĩa, Bình Ch...",700000.0,"Bình Lợi, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-70ha-m...
4424,"M&A 2 Cảng Mỹ Xuân, BRVT (P. Phú Mỹ HCM)",680000.0,"Phú Mỹ, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/ma-2-cang-my-xuan-b...
3424,"Quỹ Đất PTDA 36ha MT Nguyễn Hữu Thọ, Phước Kiể...",360000.0,"Hiệp Phước, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-36ha-m...
6345,CHUYỂN NHƯỢNG ĐẤT LÀM DỰ ÁN BỆNH VIỆN QUY MÔ 1...,290000.0,"An Khánh, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/chuyen-nhuong-dat-l...


## 3. Điền thiếu, giữ dấu vết và lưu bản cleaned

Ngoại lai được xác định bằng phương pháp thống kê IQR và tiến hành loại bỏ hoàn toàn khỏi tập dữ liệu. Quyết định này nhằm đảm bảo các biểu đồ Khám phá dữ liệu (EDA) ở bước sau không bị méo lệch phân phối, đáp ứng Tiêu chí 1 Phần C.

In [7]:
# CÁC BƯỚC ĐẦU GIỮ NGUYÊN
df_clean = df_source.copy()

for column in ['gia', 'dien_tich']:
    lower, upper = bounds[column]
    original = df_source[column]
    invalid = original.notna() & original.le(0)

    # Lưu giá trị gốc và cờ trước khi điền
    df_clean[f'{column}_goc'] = original
    df_clean[f'{column}_thieu'] = original.isna() | invalid
    
    # Đánh dấu ngoại lai
    df_clean[f'{column}_ngoai_lai'] = original.notna() & (
        (original < lower) | (original > upper)
    )
    # Giá/diện tích <= 0 không hợp lệ -> chuyển thành NaN
    df_clean.loc[invalid, column] = np.nan

# ---------------------------------------------------------
# TỐI ƯU 1: LỌC BỎ HOÀN TOÀN NGOẠI LAI TRƯỚC!
df_final = df_clean[~df_clean['gia_ngoai_lai'] & ~df_clean['dien_tich_ngoai_lai']].copy()

# TỐI ƯU 2: TÍNH MEDIAN VÀ ĐIỀN KHUYẾT TRÊN TẬP DỮ LIỆU ĐÃ SẠCH BÓNG NGOẠI LAI
for column in ['gia', 'dien_tich']:
    median_by_area = df_final.groupby('vi_tri', dropna=False)[column].transform('median')
    df_final[column] = df_final[column].fillna(median_by_area).fillna(df_final[column].median())
# ---------------------------------------------------------

# Không đoán số phòng ngủ; cờ cho biết trường này vốn bị thiếu.
df_final['so_phong_ngu_thieu'] = df_source['so_phong_ngu'].isna()

# TỐI ƯU 3: XÓA 2 CỘT CỜ NGOẠI LAI (Giải phóng bộ nhớ vì tất cả đều đã là False)
df_final = df_final.drop(columns=['gia_ngoai_lai', 'dien_tich_ngoai_lai'])

# Kiểm tra dữ liệu đầu ra
assert df_final[['gia', 'dien_tich']].notna().all().all(), 'Giá/diện tích vẫn còn thiếu.'

# Lưu bản dữ liệu đã dọn dẹp sạch sẽ
output_file = project / 'data' / 'processed' / 'cafeland_missing_outliers_cleaned.csv'
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f'Đã lọc bỏ ngoại lai và điền khuyết. Số dòng còn lại để đưa vào mô hình: {len(df_final):,} dòng')
print(f'Đã lưu thành công vào: {output_file}')

Đã lọc bỏ ngoại lai và điền khuyết. Số dòng còn lại để đưa vào mô hình: 5,435 dòng
Đã lưu thành công vào: D:\iuh\hk1_nam3\DAP\Project\data\processed\cafeland_missing_outliers_cleaned.csv
